# Module 5 — Operator Learning with DeepONet

**NHERI SimCenter/DesignSafe 2026 SPARC · Day 3, Session 3b**

**Exercise** [![Try on DesignSafe](https://raw.githubusercontent.com/DesignSafe-CI/training-ai/main/DesignSafe-Badge.svg)](https://jupyter.designsafe-ci.org/hub/user-redirect/lab/tree/CommunityData/Training/2026-SPARC/Day3/Session3b/05-deeponet-cantilever-exercise.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DesignSafe-CI/training-ai/blob/main/05-operator-learning/05-deeponet-cantilever-exercise.ipynb)

**Solution** [![Try on DesignSafe](https://raw.githubusercontent.com/DesignSafe-CI/training-ai/main/DesignSafe-Badge.svg)](https://jupyter.designsafe-ci.org/hub/user-redirect/lab/tree/CommunityData/Training/2026-SPARC/Day3/Session3b/05-deeponet-cantilever.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DesignSafe-CI/training-ai/blob/main/05-operator-learning/05-deeponet-cantilever.ipynb)

> Cantilever DeepONet example after Somdatta Goswami (Johns Hopkins).

---

## Where we are

Module 4's PINN solved the beam from the equation alone, with no labelled data.
Then we noticed the catch: it solved **one load case**. Change the load and every
weight is wrong. Minutes of optimisation per load, when a design study needs
hundreds.

Look at the pattern across the session:

| Module | Learns | Maps | New query costs |
| --- | --- | --- | --- |
| 1 Regression | 3 coefficients | $\mathbb{R}^3 \to \mathbb{R}$ | instant |
| 2 MLP | a nonlinear surrogate | $\mathbb{R}^3 \to \mathbb{R}$ | instant |
| 4 PINN | one solution field | $x \mapsto w(x)$ | **retrain (minutes)** |
| 5 DeepONet | the solution **operator** | $q(\cdot) \mapsto w(\cdot)$ | instant |

The last row is the one we build now: a single trained network that takes a
*whole load function* as input and returns the *whole deflection field*, for any
load in the family — without retraining.

## The problem

A 2 m x 0.2 m linear-elastic cantilever under a displacement-controlled boundary
condition. The applied displacement profile is drawn from a Gaussian random
field, so no two load cases look alike. For each one, a finite element solve gives
the full 2D displacement field.

<img src="figs/deeponet-cantilever-schematic.png" width="900"/>

We have **1000** such (load, field) pairs. We want the map between them.

## What you will do

| Part | |
| --- | --- |
| 1 | From functions to operators — and why an MLP struggles |
| 2 | The operator universal approximation theorem |
| 3 | Build and train a DeepONet |
| 4 | Inspect the **learned basis** — the satisfying part |
| 5 | Zero-shot prediction, and the limits |

## Setup

In [ ]:
%pip install torch scipy matplotlib numpy --quiet

In [ ]:
%matplotlib inline

import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.io import loadmat

RNG = 42
torch.manual_seed(RNG)
np.random.seed(RNG)

plt.rcParams.update({"figure.figsize": (11, 4), "font.size": 11})
MAT_NAME = "cantilever_beam_deflection.mat"


def find_mat():
    """The 20 MB dataset: Community Data, then local, then GitHub."""
    here = Path.cwd()
    for c in [Path("/home/jupyter/CommunityData/Training/2026-SPARC/Day3/Session3b") / MAT_NAME,
              here / MAT_NAME,
              here / "05-operator-learning" / MAT_NAME,
              here.parent / "training-deeponet" / MAT_NAME]:
        if c.exists():
            print(f"using {c}")
            return str(c)
    import urllib.request
    print("downloading 20 MB from GitHub (one time)...")
    urllib.request.urlretrieve("https://raw.githubusercontent.com/DesignSafe-Training/deeponet/refs/heads/main/cantilever_beam_deflection.mat", MAT_NAME)
    return MAT_NAME


print("torch", torch.__version__)

## The data

In [ ]:
raw = loadmat(find_mat())

# TODO: pull out app_disp (input functions), coord_x/coord_y (query points),
#       disp_x/disp_y (output fields), sensor_loc_disp (sensor positions).
#       Print the shapes and work out what each axis means.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

for i in range(6):
    ax[0].plot(sensor_loc, u_all[i], lw=1.5, alpha=.85)
ax[0].set(xlabel="x (m)", ylabel="applied displacement",
          title=f"6 of {N_SAMPLES} input functions (GRF draws)")
ax[0].grid(alpha=.3)

for a, comp, name in ((ax[1], sy_all[0], "$u_y$ (case 0)"),
                      (ax[2], sy_all[1], "$u_y$ (case 1)")):
    sc = a.scatter(xy[:, 0], xy[:, 1], c=comp, s=7, cmap="RdBu_r")
    a.set(xlabel="x (m)", ylabel="y (m)", title=f"Output field {name}")
    a.set_aspect("equal")
    plt.colorbar(sc, ax=a, fraction=.025)

plt.tight_layout(); plt.show()

Each input is a *function* — a displacement profile along the beam — not a
handful of scalars. Each output is a *field* on 1314 nodes. That is the shape of
the problem operator learning is built for.

## Part 1 — From functions to operators

Modules 2 and 3 approximated **functions**: $f: \mathbb{R}^d \to \mathbb{R}$, a
finite vector in, a number out. An **operator** maps a function to a function:

$$\mathcal{G}: \mathcal{U} \to \mathcal{S}, \qquad
u(\cdot) \;\longmapsto\; s(\cdot) = \mathcal{G}(u)(\cdot)$$

Our target is the solution operator of the beam: given the applied displacement
profile $u$, return the displacement field $s$.

<img src="figs/operator_concept.png" width="760"/>

### The immediate difficulty

A function lives in an infinite-dimensional space. Networks take finite vectors.

The standard resolution is to **discretise the input**: evaluate $u$ at $m$ fixed
sensor locations and feed the vector $[u(x_1), \dots, u(x_m)]$. Our data already
does this, with $m = 100$.

Note the asymmetry, because it matters later: the *input* is pinned to those 100
sensors, but the *output* can be queried at any $(x, y)$ we like. A DeepONet is
mesh-free in its output and fixed-grid in its input. (Fourier Neural Operators
relax the input side too — see Part 5.)

### Why not just concatenate everything into one MLP?

The obvious baseline: build a 102-dimensional input $[u_1, \dots, u_{100}, x, y]$
and map it to $(u_x, u_y)$ with a plain MLP. Nothing stops you. Let's find out
whether DeepONet's structure earns its place, rather than assuming it.

In [ ]:
# --- split, normalise (statistics from the training split only) ---
N_TRAIN = 800
idx_tr, idx_te = np.arange(N_TRAIN), np.arange(N_TRAIN, N_SAMPLES)

s_all = np.stack([sx_all, sy_all], axis=-1)          # (1000, 1314, 2)

u_mu, u_sd = u_all[idx_tr].mean(), u_all[idx_tr].std()
s_sd = s_all[idx_tr].reshape(-1, 2).std(0)           # per component
xy_lo, xy_hi = xy.min(0), xy.max(0)

T = lambda a: torch.tensor(a, dtype=torch.float32)
U_tr, U_te = T((u_all[idx_tr] - u_mu) / u_sd), T((u_all[idx_te] - u_mu) / u_sd)
S_tr, S_te = T(s_all[idx_tr] / s_sd), T(s_all[idx_te] / s_sd)
XY = T(2 * (xy - xy_lo) / (xy_hi - xy_lo) - 1)       # -> [-1, 1]^2

print(f"train {len(U_tr)} cases   test {len(U_te)} cases")
print(f"output std per component: u_x {s_sd[0]:.4f}, u_y {s_sd[1]:.4f}")
print("y spans 0.2 m and x spans 2 m, so the coordinates are rescaled to "
      "[-1,1] -- otherwise the trunk barely sees y")

In [ ]:
def mlp(sizes, act=nn.Tanh):
    """TODO: plain fully-connected stack helper."""
    ...


def rel_l2(pred, true):
    """TODO: per-case relative L2 error (%)."""
    ...


class NaiveMLP(nn.Module):
    """TODO: concatenate the 100 sensor values with (x,y) -> 102 inputs,
    map to (u_x, u_y). You will need to broadcast U across query points.
    """
    ...


def train_operator(model, epochs=400, lr=1e-3, bs=64, n_query=None,
                   eval_every=20, quiet=False, tag=""):
    """TODO: minibatch Adam over load cases; record train and test MSE.

    Two things that matter for wall-clock:
      - n_query: if set, sample that many query points per step instead of
        all 1314. The naive model needs this; work out why.
      - eval_every: a full test-set pass is expensive, so do it periodically
        and record which epoch each measurement came from.
    """
    ...

Keep that number. The DeepONet will get the same parameter budget, the
same optimiser, and the same number of epochs.

### One detail that is not a detail: `n_query=128`

The naive model had to be trained on a random **subset** of 128 query points per
step, while the DeepONet below trains on all 1314 every step. That is not us
handicapping the baseline — it is the baseline being unaffordable otherwise.

The reason is structural. To predict at $Q$ points, the naive model must push
the 100 sensor values through its layers $Q$ separate times, because they are
concatenated with the coordinates before the first layer. At 1314 nodes that is
1314 copies of the same input function through the same weights.

We measure exactly how much that costs in Part 3, once we have something to
compare it against.

## Part 2 — The operator universal approximation theorem

Module 2 leaned on Cybenko/Hornik: networks are dense in $C(K)$. There is an
operator analogue, and it is older than you might expect.

> **Theorem (Chen & Chen, 1995).** Let $\sigma$ be a continuous non-polynomial
> activation, $K_1 \subset \mathbb{R}^d$ and $V \subset C(K_1)$ compact, and
> $\mathcal{G}: V \to C(K_2)$ a continuous operator. Then for any
> $\varepsilon > 0$ there are $m$ sensor points $x_j$, integers $p$, and network
> weights such that
> $$\left|\; \mathcal{G}(u)(y) \;-\; \sum_{k=1}^{p}
> \underbrace{b_k\!\left(u(x_1), \dots, u(x_m)\right)}_{\text{branch}}
> \cdot \underbrace{t_k(y)}_{\text{trunk}} \;\right| < \varepsilon$$
> for all $u \in V$ and all $y \in K_2$.

### Read the formula as an architecture

That sum **is** DeepONet. Lu et al. (2019) took the theorem literally and made
both factors neural networks:

$$\mathcal{G}_\theta(u)(y) = \sum_{k=1}^{p} b_k(u)\, t_k(y) + b_0$$

| Piece | Input | Output | Role |
| --- | --- | --- | --- |
| **Branch** net | $u$ at $m$ sensors | $p$ coefficients | encodes *which* load case this is |
| **Trunk** net | a query point $y$ | $p$ basis values | builds a basis over the domain |
| Inner product | | one field value | combines them |

<img src="figs/deeponet-arch.png" width="720"/>

### Why this factorisation is the whole point

The trunk depends only on $y$, never on $u$. So it learns **one shared set of
basis functions** for the entire family of load cases, and the branch only has to
say how much of each basis function this particular load needs.

That should look familiar. It is the structure of a modal or spectral expansion —
$w(x) = \sum_k c_k \phi_k(x)$ — except that the basis $\phi_k$ is *learned from
data* rather than assumed. In Part 4 we plot the $t_k$ and compare them to what a
structural engineer would expect.

Practical consequences:

- The trunk is evaluated **once** for all query points, then reused for every load
  case in the batch. That is what the naive MLP could not do.
- Querying a new point is a trunk forward pass — the output is mesh-free.
- You get an interpretable object out: a basis.

## Part 3 — Implementation

Two subtleties.

**Two output components.** We need $u_x$ and $u_y$. Rather than two separate
networks, have branch and trunk each emit $2p$ values and read them as
$2 \times p$ — one set of coefficients and one basis per component.

**The contraction.** With a batch of $B$ load cases and $Q$ query points we want
a $(B, Q, 2)$ output from a $(B, 2, p)$ branch and a $(Q, 2, p)$ trunk. That is
one `einsum`:

```python
torch.einsum("bop,qop->bqo", branch_out, trunk_out)
```

summing over the basis index `p` while keeping batch `b`, query `q`, and
component `o` — no loops, no broadcasting by hand.

In [ ]:
class DeepONet(nn.Module):
    """TODO: implement G(u)(y) = sum_k b_k(u) * t_k(y) + bias.

    - branch: m inputs  -> p * n_out outputs, reshaped to (B, n_out, p)
    - trunk:  2 inputs  -> p * n_out outputs, reshaped to (Q, n_out, p)
    - combine with torch.einsum("bop,qop->bqo", ...) and add a bias
    Also add .basis(XY) and .coeffs(U) helpers -- Part 4 needs them.
    """
    ...


torch.manual_seed(RNG)
don = DeepONet(p=100, width=128, depth=5)

### First, the cost claim from Part 1, measured

Same batch, same number of query points, one training step each.

In [ ]:
# TODO: time one training step of each model at 128, 512, and all 1314
#       query points. Why does the naive model's cost grow so much faster?

That ratio is the architecture's justification. It is also why the naive
baseline had to be trained on 128 sampled points while the DeepONet trains on all
1314 — so bear in mind the accuracy comparison below actually *favours* the
baseline, since it is the one that got the cheaper problem.

### Train it

Same optimiser, same epochs, same batch size as the naive baseline — but all 1314
query points per step.

In [ ]:
h_don = train_operator(don, epochs=400, tag="deeponet")

with torch.no_grad():
    pred_te = don(U_te, XY).numpy() * s_sd
true_te = S_te.numpy() * s_sd
r_don = rel_l2(pred_te, true_te)

print(f"\n{'model':<12s} {'params':>10s} {'seconds':>9s} {'mean rel L2':>13s} {'worst':>9s}")
print(f"{'naive MLP':<12s} {sum(p.numel() for p in naive.parameters()):>10,} "
      f"{h_naive['seconds']:>9.0f} {r_naive.mean():>12.2f}% {r_naive.max():>8.2f}%")
print(f"{'DeepONet':<12s} {n_don:>10,} {h_don['seconds']:>9.0f} "
      f"{r_don.mean():>12.2f}% {r_don.max():>8.2f}%")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))

ax[0].semilogy(h_naive["train"], label="naive MLP — train", alpha=.75)
ax[0].semilogy(h_naive["test_epoch"], h_naive["test"], "o--", ms=3,
               label="naive MLP — test", alpha=.75)
ax[0].semilogy(h_don["train"], label="DeepONet — train")
ax[0].semilogy(h_don["test_epoch"], h_don["test"], "o--", ms=3,
               label="DeepONet — test")
ax[0].set(xlabel="epoch", ylabel="MSE (normalised)", title="Same budget, both models")
ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)

ax[1].hist([r_naive, r_don], bins=25, label=["naive MLP", "DeepONet"])
ax[1].set(xlabel="relative $L_2$ error (%)", ylabel="test cases",
          title="Per-case accuracy on 200 unseen loads")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

Both curves sit almost on top of their training counterparts — with 800
load cases and no label noise, neither model is overfitting (exactly the point
Module 2 made about noise).

### A test case, in full

In [ ]:
worst = int(np.argmax(r_don))
best = int(np.argmin(r_don))

for case, label in ((best, "best"), (worst, "worst")):
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.1))
    t, p = true_te[case, :, 1], pred_te[case, :, 1]        # u_y component
    vmax = np.abs(t).max()
    for a, val, title, cmap, lim in (
            (ax[0], t, "FE truth ($u_y$)", "RdBu_r", vmax),
            (ax[1], p, "DeepONet ($u_y$)", "RdBu_r", vmax),
            (ax[2], np.abs(p - t), "absolute error", "magma", None)):
        sc = a.scatter(xy[:, 0], xy[:, 1], c=val, s=7, cmap=cmap,
                       vmin=-lim if lim else None, vmax=lim if lim else None)
        a.set(title=title, xlabel="x (m)", ylabel="y (m)")
        a.set_aspect("equal")
        plt.colorbar(sc, ax=a, fraction=.025)
    fig.suptitle(f"{label} test case (#{case}) — relative $L_2$ = "
                 f"{r_don[case]:.2f}%", y=1.06)
    plt.tight_layout(); plt.show()

## Part 4 — The learned basis

This is the part worth the module.

The trunk never sees a load case. It is a function of position only, so the $p$
functions $t_k(x, y)$ form a **basis for every deflection field the operator can
produce**. The branch merely picks coefficients.

So: what basis did it discover?

<img src="figs/basis_analysis.png" width="720"/>

In [ ]:
# TODO: extract the trunk basis with don.basis(XY) and the branch
#       coefficients with don.coeffs(U_te).
#       Rank the modes by  std(coefficient) * ||basis function||  --
#       a mode with a large basis but a coefficient that never varies
#       is doing nothing. How many modes carry 90% of the total?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 4.4))
for r, (a, k) in enumerate(zip(axes.ravel(), order[:8])):
    b = basis[:, 1, k]
    lim = np.abs(b).max()
    a.scatter(xy[:, 0], xy[:, 1], c=b, s=5, cmap="RdBu_r", vmin=-lim, vmax=lim)
    a.set_title(f"mode {k} (rank {r+1})", fontsize=10)
    a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
plt.suptitle(r"Learned trunk basis for $u_y$ — nobody told it about mode shapes",
             y=1.02)
plt.tight_layout(); plt.show()

Smooth, structured, and clearly organised by spatial frequency — with no
beam mode shape, polynomial, or Fourier term ever supplied.

But look again at the contribution numbers above. They are **flat**: the top mode
scores about 3.0 and the eighth still about 1.8, and it took 82 of the 100 modes
to reach 90% of the total. That is not what a good basis looks like. Let's find
out what is going on, because the answer is the most useful thing in this
module.

### How many dimensions does this problem actually have?

Ask the data, independently of the network. A POD (equivalently, an SVD) of the
800 training fields gives the intrinsic dimensionality of the solution
manifold.

In [ ]:
from scipy.linalg import subspace_angles

# TODO: run an SVD/POD on the 800 training fields (u_y component) and report
#       how many modes carry 90 / 99 / 99.9% of the field energy.
#       How does that compare with the p = 100 modes we gave the trunk?

Six modes for 99% of the energy. The solution manifold really is
low-dimensional — so the flat contribution spectrum is not the *problem* being
high-rank. It is the network's basis being wasteful.

### Did the trunk find the right subspace?

Two separate questions, and they have different answers:

1. Does the trunk **span** the space the POD modes span?
2. Are the trunk's individual modes a **good basis** for it?

In [ ]:
# TODO: SVD the trunk basis matrix T_y = basis[:, 1, :].
#       (a) What is its condition number?
#       (b) Compute the Gram matrix and look at the off-diagonal cosines --
#           how far from orthogonal are the 100 modes?
#       (c) Use scipy.linalg.subspace_angles to compare the leading k
#           directions of the trunk against the leading k POD modes,
#           for k = 2, 4, 6, 8. Where does the agreement break down?

Both answers at once.

**Yes, it found the space — the important part of it.** The trunk's leading four
directions sit within about 12 degrees of the four leading POD modes, and those
four carry ~93% of the field energy. The network independently rediscovered the
dominant deformation modes of a cantilever from 800 solutions. That is a real
result and it is why the pictures above look like mode shapes.

**No, it is not a good basis.** Three measurements say so:

- **Non-orthogonal.** The mean absolute cosine between distinct trunk modes is
  around 0.4. POD modes are orthogonal by construction; these overlap heavily and
  encode the same information many times over.
- **Catastrophically ill-conditioned.** A condition number in the hundreds of
  thousands. The 100 modes are very nearly linearly dependent.
- **Unordered past the leading few.** At $k = 6$ one principal angle jumps to
  above 80 degrees, and it only gets worse from there. The trunk's sixth direction
  has nothing to do with the POD's sixth. Nothing in the loss asked for an
  ordering, so there isn't one.

### The consequence: you cannot truncate it

POD comes with the Eckart-Young guarantee — keep the top $k$ modes and you have
the provably best rank-$k$ approximation of the data. A DeepONet trunk comes with
no such promise, and here is what that costs. (The two columns are not measuring
identical things: POD reconstructs the *training* fields it was computed from,
while the DeepONet predicts *unseen* ones. Read the shapes of the two curves, not
the gap at any single $k$.)

In [ ]:
def truncated_pred(model, U, XY, keep):
    """Reconstruct using only the `keep` highest-contribution trunk modes."""
    B = model.coeffs(U).numpy()
    Tk = model.basis(XY).numpy()
    sel = order[:keep]
    out = np.einsum("bop,qop->bqo", B[:, :, sel], Tk[:, :, sel])
    return (out + model.bias.detach().numpy()) * s_sd


keeps = [1, 2, 5, 10, 20, 50, don.p]
errs = [rel_l2(truncated_pred(don, U_te, XY, k), true_te).mean() for k in keeps]

pod_errs = []
for k in keeps:
    coef = (Y - Y_mean) @ U_pod[:, :k]
    rec = coef @ U_pod[:, :k].T + Y_mean
    pod_errs.append(np.linalg.norm(rec - Y) / np.linalg.norm(Y) * 100)

print(f"  {'k':>5s} {'DeepONet top-k':>16s} {'POD top-k':>12s}")
for k, e, pe in zip(keeps, errs, pod_errs):
    print(f"  {k:>5d} {e:>15.2f}% {pe:>11.2f}%")

plt.figure(figsize=(6.8, 4.2))
plt.semilogx(keeps, errs, "o-", label="DeepONet trunk, top-$k$ by contribution")
plt.semilogx(keeps, pod_errs, "s-", label="POD, top-$k$ (optimal)")
plt.xlabel("modes retained"); plt.ylabel("relative $L_2$ error (%)")
plt.title("POD truncates gracefully. The learned basis does not.")
plt.legend(fontsize=9); plt.grid(True, which="both", alpha=.3)
plt.tight_layout(); plt.show()

POD falls below 1% by 10 modes and keeps collapsing. The DeepONet basis,
truncated the same way, is still ~35% wrong at 50 of 100 modes and only becomes
usable when essentially all of them are kept — because the information is smeared
across every mode, and dropping any destroys cancellations the network relied on.

**So: a DeepONet is a learned spectral method with an unmanaged basis.** It finds
the right subspace and represents it wastefully. That matters when you want to
compress the model, interpret the modes beyond the leading few, or trust the mode
count as a rank estimate.

Three standard fixes, in increasing order of effort:

- **Orthogonalise after training.** A QR or SVD of the trunk matrix, as we just
  did, gives an ordered orthonormal basis for the same span — free, and enough for
  interpretation.
- **Penalise non-orthogonality during training.** Add $\|T^\top T - I\|^2$ to the
  loss and the trunk comes out closer to POD-like.
- **POD-DeepONet.** Skip the learning entirely for the trunk: compute POD modes
  from the training fields and let the branch learn only the coefficients. Often
  more accurate *and* smaller, at the cost of a fixed output basis.

The honest summary is that the basis picture is the most interesting thing a
DeepONet gives you and the part most in need of scrutiny. A pretty mode plot is
not evidence of a well-conditioned model.

## Part 5 — Zero-shot prediction, and the limits

The claim that started the module: a new load case costs one forward pass, not a
retrain. Let's price it.

In [ ]:
new_cases = U_te[:32]

t0 = time.time()
with torch.no_grad():
    _ = don(new_cases, XY)
t_infer = time.time() - t0

print(f"  32 unseen load cases, full 1314-node field each")
print(f"    DeepONet inference : {t_infer*1000:>9.1f} ms   "
      f"({t_infer/32*1000:.2f} ms per case)")
print(f"    DeepONet training  : {h_don['seconds']:>9.1f} s    (once, amortised)")
print()
print("  For comparison, from Module 4: a PINN needs a full retrain per load")
print("  case -- seconds to minutes each, and it never gets cheaper.")
print(f"\n  Speedup for a 1000-case design study: roughly "
      f"{1000 * 4.0 / (h_don['seconds'] + 1000 * t_infer / 32):,.0f}x")
print("  (taking 4 s per PINN solve, the fast end of what we measured)")

### Querying points that are not FE nodes

The trunk takes any $(x, y)$, so we can evaluate the field on a grid the mesh
never had. This is the mesh-free property, and it is genuinely useful for
post-processing.

In [ ]:
gx, gy = np.meshgrid(np.linspace(0, 2, 160), np.linspace(0, 0.2, 24))
grid = np.stack([gx.ravel(), gy.ravel()], 1)
XY_grid = T(2 * (grid - xy_lo) / (xy_hi - xy_lo) - 1)

with torch.no_grad():
    on_grid = (don(U_te[:1], XY_grid).numpy() * s_sd)[0, :, 1]

fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
ax[0].scatter(xy[:, 0], xy[:, 1], c=true_te[0, :, 1], s=8, cmap="RdBu_r")
ax[0].set(title=f"FE truth on {N_NODES} nodes", xlabel="x (m)", ylabel="y (m)")
im = ax[1].contourf(gx, gy, on_grid.reshape(gx.shape), levels=30, cmap="RdBu_r")
ax[1].set(title=f"DeepONet on a {gx.size:,}-point grid it never saw",
          xlabel="x (m)", ylabel="y (m)")
for a in ax:
    a.set_aspect("equal")
plt.colorbar(im, ax=ax[1], fraction=.025)
plt.tight_layout(); plt.show()

### The honest limits

**The training data is the expensive part.** We used 800 finite element solves.
DeepONet moved the cost from query time to a one-off offline campaign — it did not
remove it. The economics only work if you will make many queries.

**It only knows the family it was trained on.** Our loads were Gaussian random
field draws with a particular length scale. Feed it a point load, or a GRF with a
much shorter correlation length, and it will extrapolate — badly, and without
warning. This is the UAT domain caveat from Module 2, now applying to a space of
*functions*, where "outside the training distribution" is much harder to detect.

**The input grid is fixed.** Change your sensor layout and the branch network is
invalid. Fourier Neural Operators address exactly this by working in Fourier
space, giving discretisation invariance — train at one resolution, evaluate at
another.

**No physics is enforced.** This is a purely data-driven fit. Nothing in the loss
requires equilibrium, and the predicted field can violate it.

### Closing the circle: physics-informed DeepONet

That last limitation has an obvious fix given where we have been. Put the PDE
residual from Module 4 into the operator loss of Module 5:

$$\mathcal{L} = \underbrace{\|\mathcal{G}_\theta(u) - s\|^2}_{\text{data, Module 5}}
+ \lambda \underbrace{\|\mathcal{N}[\mathcal{G}_\theta(u)]\|^2}_{\text{PDE residual, Module 4}}$$

Differentiate the trunk with respect to its inputs — the same `create_graph=True`
trick from Module 4 — and you can evaluate the residual at any point, for any
input function. With enough physics weight you can train an operator with **very
few or no labelled solutions**.

<img src="figs/pideeponet-arch.png" width="720"/>

That is the synthesis of this whole session: Module 2's networks, Module 4's
residuals, Module 5's operator structure, in one model.

## Summary

| | |
| --- | --- |
| **Operators map functions to functions** | Discretise the input at $m$ sensors; the output stays mesh-free. |
| **Chen & Chen (1995) gives the architecture** | $\mathcal{G}(u)(y) \approx \sum_k b_k(u)\,t_k(y)$ — branch times trunk, read literally. |
| **The factorisation is the efficiency** | The trunk is computed once per batch and shared across all query points; the naive MLP repeats the input 1314 times. |
| **The trunk is a learned basis** | Smooth, increasing-frequency modes that nobody specified — a learned spectral method, and a low-rank diagnostic. |
| **Queries become free, training does not** | 800 FE solves offline buys millisecond inference. Worth it only for many-query workflows. |
| **It knows only its training family** | Out-of-distribution input *functions* fail silently. |
| **PI-DeepONet closes the loop** | Add Module 4's residual to Module 5's loss and shed the labelled data. |

### The session in one line

**Module 1** fit a formula. **Module 2** replaced the formula with a network when
we no longer knew the right features. **Module 3** interrogated the network.
**Module 4** replaced the data with physics. **Module 5** replaced one solution
with the solution operator.

### Go deeper

- [SciML — DeepONet from scratch](https://kks32-courses.github.io/sciml/02-deeponet/deeponet.html)
- [SciML — physics-informed DeepONet](https://kks32-courses.github.io/sciml/02-deeponet/pideeponet.html)
- [SciML — how to read a DeepONet](https://kks32-courses.github.io/sciml/02-deeponet/deeponet-explanation.html)
- [SciML — the batched einsum, in detail](https://kks32-courses.github.io/sciml/02-deeponet/einsum.html)
- [SciML — Fourier Neural Operators](https://kks32-courses.github.io/sciml/05-fno/fno.html)
- [SciML — function encoders](https://kks32-courses.github.io/sciml/03-function-encoder/03-function-encoder.html)
- [DesignSafe DeepONet training](https://github.com/DesignSafe-Training/deeponet) — the original JAX/Flax version of this example, by Somdatta Goswami
- Lu, Jin & Karniadakis (2019), *DeepONet* · Chen & Chen (1995), the operator UAT ·
  Li et al. (2020), *Fourier Neural Operator*